### Vector Store - FAISS
Facebook AI Similarity Search (FAISS) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation anf parameter tuning.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader('Speech.txt')
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=100,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


Created a chunk of size 304, which is longer than the specified 100


In [5]:
docs

[Document(metadata={'source': 'Speech.txt'}, page_content='My parents impressed on me the value of that you work hard for what you want in life. That your word is your bond and you do what you say and keep your promise. That you treat people with respect. Show the values and morals in in the daily life. That is the lesson that we continue to pass on to our son.'),
 Document(metadata={'source': 'Speech.txt'}, page_content='We need to pass those lessons on to the many generations to follow. [Cheering] Because we want our children in these nations to know that the only limit to your achievement is the strength of your dreams and your willingness to work for them.')]

In [7]:
embeddings=OllamaEmbeddings(model="gemma2:2b")
db=FAISS.from_documents(docs,embeddings)
db

In [9]:
### querying
query="What is the speech about?"
docs=db.similarity_search(query)
docs[0].page_content

'My parents impressed on me the value of that you work hard for what you want in life. That your word is your bond and you do what you say and keep your promise. That you treat people with respect. Show the values and morals in in the daily life. That is the lesson that we continue to pass on to our son.'

### As a Retriever
We can also convert the vectorstore nto a Retriever class. This allows us to easily use it in other Langchain methods, which largely work with retrievers.

In [12]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'My parents impressed on me the value of that you work hard for what you want in life. That your word is your bond and you do what you say and keep your promise. That you treat people with respect. Show the values and morals in in the daily life. That is the lesson that we continue to pass on to our son.'

### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which you to return not only the documents but also the distance score of the query of them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [13]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='5214f976-8a7d-48af-868a-7bd4bd835b63', metadata={'source': 'Speech.txt'}, page_content='My parents impressed on me the value of that you work hard for what you want in life. That your word is your bond and you do what you say and keep your promise. That you treat people with respect. Show the values and morals in in the daily life. That is the lesson that we continue to pass on to our son.'),
  np.float32(4073.4712)),
 (Document(id='dc6439aa-bf25-4286-a223-d3a21cb9fa42', metadata={'source': 'Speech.txt'}, page_content='We need to pass those lessons on to the many generations to follow. [Cheering] Because we want our children in these nations to know that the only limit to your achievement is the strength of your dreams and your willingness to work for them.'),
  np.float32(4380.631))]

In [14]:
embedding_vector=embeddings.embed_query(query)
embedding_vector

[-1.0422042608261108,
 1.346085548400879,
 -1.126206398010254,
 1.5257503986358643,
 0.8251509070396423,
 0.6241821646690369,
 -1.5133168697357178,
 -0.6532307863235474,
 1.4555606842041016,
 -0.041945680975914,
 -0.9051247239112854,
 -0.32315167784690857,
 -1.2828322649002075,
 -0.7995197176933289,
 -0.620638370513916,
 -2.2405736446380615,
 -0.29566094279289246,
 0.08169962465763092,
 -2.4147536754608154,
 -0.14798900485038757,
 0.8945407867431641,
 0.6157183051109314,
 -1.4160568714141846,
 1.0769199132919312,
 -0.10155699402093887,
 0.29004690051078796,
 1.6417075395584106,
 -0.29894885420799255,
 -1.2623891830444336,
 0.5998660326004028,
 0.4258185625076294,
 -1.518807053565979,
 0.7613455653190613,
 1.1818969249725342,
 0.6531196236610413,
 2.0561821460723877,
 -0.805570662021637,
 -0.0016094864113256335,
 1.2780357599258423,
 -1.5932493209838867,
 -0.20705723762512207,
 1.3226910829544067,
 1.4455687999725342,
 0.13148771226406097,
 -1.3347415924072266,
 -1.527916669845581,
 0.2

In [15]:
docs_and_score=db.similarity_search_by_vector(embedding_vector)
docs_and_score

[Document(id='5214f976-8a7d-48af-868a-7bd4bd835b63', metadata={'source': 'Speech.txt'}, page_content='My parents impressed on me the value of that you work hard for what you want in life. That your word is your bond and you do what you say and keep your promise. That you treat people with respect. Show the values and morals in in the daily life. That is the lesson that we continue to pass on to our son.'),
 Document(id='dc6439aa-bf25-4286-a223-d3a21cb9fa42', metadata={'source': 'Speech.txt'}, page_content='We need to pass those lessons on to the many generations to follow. [Cheering] Because we want our children in these nations to know that the only limit to your achievement is the strength of your dreams and your willingness to work for them.')]

In [17]:
## Saving and Loading FAISS Vectorstore
db.save_local("faiss_index")


In [20]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [21]:
docs

[Document(id='5214f976-8a7d-48af-868a-7bd4bd835b63', metadata={'source': 'Speech.txt'}, page_content='My parents impressed on me the value of that you work hard for what you want in life. That your word is your bond and you do what you say and keep your promise. That you treat people with respect. Show the values and morals in in the daily life. That is the lesson that we continue to pass on to our son.'),
 Document(id='dc6439aa-bf25-4286-a223-d3a21cb9fa42', metadata={'source': 'Speech.txt'}, page_content='We need to pass those lessons on to the many generations to follow. [Cheering] Because we want our children in these nations to know that the only limit to your achievement is the strength of your dreams and your willingness to work for them.')]